In [73]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score,precision_score,f1_score,recall_score,roc_auc_score,confusion_matrix,classification_report
import warnings
warnings.filterwarnings('ignore')

In [52]:
df=pd.read_csv('../data/processed/live_data.csv')

In [53]:
df.head()

,match_id,innings,team_runs,team_balls,team_wicket,runs_target,batting_team,bowling_team,venue,over,ball,runs_left,balls_left,wickets_left,current_rr,required_rr,match_won_by
0,335982,2,1,1,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,1,222.0,119,10,6.0,11.193277,Kolkata Knight Riders
1,335982,2,2,1,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,2,221.0,119,10,12.0,11.142857,Kolkata Knight Riders
2,335982,2,2,2,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,2,221.0,118,10,6.0,11.237288,Kolkata Knight Riders
3,335982,2,3,3,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,3,220.0,117,10,6.0,11.282051,Kolkata Knight Riders
4,335982,2,4,4,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,4,219.0,116,10,6.0,11.327586,Kolkata Knight Riders


In [54]:
df['chasing_team_won']=(df['batting_team']==df['match_won_by']).astype('int64')
df.drop(columns=['match_won_by'],inplace=True)
df.head()

,match_id,innings,team_runs,team_balls,team_wicket,runs_target,batting_team,bowling_team,venue,over,ball,runs_left,balls_left,wickets_left,current_rr,required_rr,chasing_team_won
0,335982,2,1,1,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,1,222.0,119,10,6.0,11.193277,0
1,335982,2,2,1,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,2,221.0,119,10,12.0,11.142857,0
2,335982,2,2,2,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,2,221.0,118,10,6.0,11.237288,0
3,335982,2,3,3,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,3,220.0,117,10,6.0,11.282051,0
4,335982,2,4,4,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,4,219.0,116,10,6.0,11.327586,0


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135642 entries, 0 to 135641
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   match_id          135642 non-null  int64  
 1   innings           135642 non-null  int64  
 2   team_runs         135642 non-null  int64  
 3   team_balls        135642 non-null  int64  
 4   team_wicket       135642 non-null  int64  
 5   runs_target       135642 non-null  float64
 6   batting_team      135642 non-null  object 
 7   bowling_team      135642 non-null  object 
 8   venue             135642 non-null  object 
 9   over              135642 non-null  int64  
 10  ball              135642 non-null  int64  
 11  runs_left         135642 non-null  float64
 12  balls_left        135642 non-null  int64  
 13  wickets_left      135642 non-null  int64  
 14  current_rr        135642 non-null  float64
 15  required_rr       135642 non-null  float64
 16  chasing_team_won  13

In [56]:
teams_to_remove = [
    'Deccan Chargers',
    'Pune Warriors',
    'Gujarat Lions',
    'Rising Pune Supergiant',
    'Kochi Tuskers Kerala'
]
df = df[
    (~df['batting_team'].isin(teams_to_remove)) &(~df['bowling_team'].isin(teams_to_remove))
]
import numpy as np

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

In [57]:
df.head()

,match_id,innings,team_runs,team_balls,team_wicket,runs_target,batting_team,bowling_team,venue,over,ball,runs_left,balls_left,wickets_left,current_rr,required_rr,chasing_team_won
0,335982,2,1,1,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,1,222.0,119,10,6.0,11.193277,0
1,335982,2,2,1,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,2,221.0,119,10,12.0,11.142857,0
2,335982,2,2,2,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,2,221.0,118,10,6.0,11.237288,0
3,335982,2,3,3,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,3,220.0,117,10,6.0,11.282051,0
4,335982,2,4,4,0,223.0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,0,4,219.0,116,10,6.0,11.327586,0


In [65]:
X=df.drop(columns=['chasing_team_won','match_id'])
y=df['chasing_team_won']
print(y.shape)

(114581,)


In [59]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [60]:
oe=OneHotEncoder()
columns=['batting_team','bowling_team','venue']
preprocessor=ColumnTransformer(
    transformers=[('encode',OneHotEncoder(handle_unknown='ignore'),columns)],remainder='passthrough'
)

In [61]:
X_train_processed=preprocessor.fit_transform(X_train)
X_test_processed=preprocessor.transform(X_test)

In [62]:
models={
    'logisticregression':LogisticRegression(),
    'RandomForest':RandomForestClassifier(),
    'xgboost':XGBClassifier()
}

In [74]:
result=[]
best_model=None
best_auc=0
best_name=""
for name,model in models.items():
    model.fit(X_train_processed,y_train)
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:,1]
    accuracy=accuracy_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    f1=f1_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    roc=roc_auc_score(y_test,y_prob)
    result.append([name, accuracy, precision, recall, f1, roc])
    print('\n\n')
    print("\t------------",name,"-------------")
    print('printing the metrics......\n')
    print("accuracy",accuracy)
    print("f1_score:",f1)
    print("recall:",recall)
    print("precision:",precision)
    print("Confusion_matrix:\n",confusion_matrix(y_test, y_pred))
    print("Classification_report:\n",classification_report(y_test, y_pred))
    if roc>best_auc:
        best_auc = roc
        best_model = model
        best_name = name
        best_acc = accuracy
print("\n===========================")
print("Best Model :", best_name)
print("Accuracy   :", best_acc)
print("ROC-AUC    :", best_auc)
print("===========================")




	------------ logisticregression -------------
printing the metrics......

accuracy 0.8019810620936423
f1_score: 0.8105377421509686
recall: 0.8272541332878813
precision: 0.7944835488623343
Confusion_matrix:
 [[8672 2511]
 [2027 9707]]
Classification_report:
               precision    recall  f1-score   support

           0       0.81      0.78      0.79     11183
           1       0.79      0.83      0.81     11734

    accuracy                           0.80     22917
   macro avg       0.80      0.80      0.80     22917
weighted avg       0.80      0.80      0.80     22917




	------------ RandomForest -------------
printing the metrics......

accuracy 0.9975127634507135
f1_score: 0.9975722986498573
recall: 0.9980398840974944
precision: 0.9971051511281397
Confusion_matrix:
 [[11149    34]
 [   23 11711]]
Classification_report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     11183
           1       1.00      1.00      1.0

In [106]:
import joblib
joblib.dump(best_model,"../models/live_pred_model.pkl")
joblib.dump(preprocessor,"../models/live_pred_preprocessor.pkl")
print("Saved")

Saved


In [90]:
metric_data=pd.DataFrame(result,columns=['Model','accuracy','precision','recall','f1', 'roc']).sort_values(by='roc',ascending=False).reset_index().drop(columns=['index'])


In [94]:
print("\n===========================")
print("Best Model :", best_name)
print("Accuracy   :", best_acc)
print("ROC-AUC    :", best_auc)
print("===========================")
metric_data


Best Model : RandomForest
Accuracy   : 0.9975127634507135
ROC-AUC    : 0.9999669375377882


,Model,accuracy,precision,recall,f1,roc
0,RandomForest,0.997513,0.997105,0.998040,0.997572,0.999967
1,xgboost,0.988262,0.986506,0.990626,0.988561,0.999468
2,logisticregression,0.801981,0.794484,0.827254,0.810538,0.885817


In [104]:
innings = 2
team_runs = 151
team_balls = 100
team_wicket = 7
runs_target = 190
batting_team = "Chennai Super Kings"
bowling_team = "Mumbai Indians"
venue = "M Chinnaswamy Stadium"
runs_left = runs_target - team_runs
balls_left = 120 - team_balls
wickets_left = 10 - team_wicket
current_rr = (team_runs * 6) / team_balls if team_balls > 0 else 0
required_rr = (runs_left * 6) / balls_left if balls_left > 0 else 0
over = team_balls // 6
ball = team_balls % 6
new_predictions = pd.DataFrame({
    "innings": [innings],
    "team_runs": [team_runs],
    "team_balls": [team_balls],
    "team_wicket": [team_wicket],
    "runs_target": [runs_target],
    "batting_team": [batting_team],
    "bowling_team": [bowling_team],
    "venue": [venue],
    "over": [over],
    "ball": [ball],
    "runs_left": [runs_left],
    "balls_left": [balls_left],
    "wickets_left": [wickets_left],
    "current_rr": [current_rr],
    "required_rr": [required_rr]
})
new_predictions

,innings,team_runs,team_balls,team_wicket,runs_target,batting_team,bowling_team,venue,over,ball,runs_left,balls_left,wickets_left,current_rr,required_rr
0,2,151,100,7,190,Chennai Super Kings,Mumbai Indians,M Chinnaswamy Stadium,16,4,39,20,3,9.06,11.7


In [105]:
transform=preprocessor.transform(new_predictions)
y_pred=best_model.predict(transform)
y_prob=best_model.predict_proba(transform)
if y_pred[0]==1:
    print(f"Predicted Winner: {batting_team}")
else:
    print(f"Predicted Winner: {bowling_team}")
print(f"{batting_team} Win Probability: {y_prob[0][1]*100:.2f}%")
print(f"{bowling_team} Win Probability: {y_prob[0][0]*100:.2f}%")

Predicted Winner: Mumbai Indians
Chennai Super Kings Win Probability: 23.00%
Mumbai Indians Win Probability: 77.00%
